In [10]:
import os

from dotenv import load_dotenv
from fastembed import SparseTextEmbedding
from qdrant_client import QdrantClient, models
from qdrant_client.models import FieldCondition, Filter, MatchValue

load_dotenv()

True

In [3]:
_PORT = os.getenv("QDRANT_PORT", "6333")
_API_KEY = os.getenv("QDRANT_API_KEY")

client = QdrantClient(url=f"http://localhost:{_PORT}", api_key=_API_KEY)
client.set_sparse_model("Qdrant/bm42-all-minilm-l6-v2-attentions")

C:\Users\Yago\AppData\Local\Temp\ipykernel_27160\2898044061.py:4: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(url=f"http://localhost:{_PORT}", api_key=_API_KEY)


In [ ]:
my_collection = "primeras_pruebas"
if client.collection_exists(collection_name=my_collection):
    client.delete_collection(collection_name=my_collection)

client.create_collection(
    collection_name=my_collection,
    # vectors_config=models.VectorParams(
    #     size=4,  # Tamaño del vector de ejemplo
    #     distance=models.Distance.COSINE,
    # ),
    sparse_vectors_config={
        "keywords_arquitectura": models.SparseVectorParams(
            index=models.SparseIndexParams(
                on_disk=False  # Cambiar a True en entornos de producción con millones de datos
            ),
            modifier=models.Modifier.IDF,  # BM42 calcula la importancia combinando pesos locales e IDF global
        )
    },
)

True

In [ ]:
class KeywordEmbedder:
    def __init__(self, model_name: str = "Qdrant/bm42-all-minilm-l6-v2-attentions"):
        # Cargamos el modelo una sola vez al instanciar
        self.sparse_model = SparseTextEmbedding(model_name=model_name)

    def get_sparse_vector(self, keywords: list[str]) -> models.SparseVector:
        """
        Toma una lista de keywords, las une y genera el formato
        que Qdrant necesita para insertar o buscar.
        """
        # Convertimos la lista de keywords a un string plano (como acordamos)
        text_input = " ".join(keywords)

        # Generamos el embedding (retorna un generador, tomamos el primero)
        embedding = list(self.sparse_model.embed([text_input]))[0]

        return models.SparseVector(indices=embedding.indices, values=embedding.values)


embedder = KeywordEmbedder()


# 1. Indexing
def index_pdi(client, collection, pdi_data):
    vector = embedder.get_sparse_vector(pdi_data["keywords"])
    client.upsert(
        collection_name=collection,
        points=[
            models.PointStruct(
                id=pdi_data["id"],
                vector={"keywords_arquitectura": vector},
                payload={"nombre": pdi_data["nombre"]},
            )
        ],
    )


# 2. Retrieval
def search_similar_pdis(
    client: QdrantClient, collection_name: str, query_keywords: list[str]
) -> list[models.ScoredPoint]:
    # 1. Generamos el vector disperso de la consulta, donde obtenemos un objeto SparseVector (indices y values)
    sparse_vector = embedder.get_sparse_vector(query_keywords)

    # 2. Ejecutamos la búsqueda usando query_points (el parámetro 'using' debe coincidir con el nombre definido en sparse_vectors_config)
    search_result = client.query_points(
        collection_name=collection_name,
        query=sparse_vector,
        using="keywords_arquitectura",
        limit=5,
        with_payload=True,  # Opcional: para recibir los metadatos ("nombre")
    )

    return search_result.points

In [16]:
pdis = [
    {
        "id": 261274,
        "nombre": "David Camacho Fernandez",
        "keywords": [
            "deep learning",
            "machine learning",
            "clustering",
            "social network analysi",
            "ant colony optimization",
            "multimodal",
            "genetic algorithm",
            "transformer",
            "seriou game",
            "genetic programming",
            "mission planning",
            "twitter",
            "data mining",
            "feature selection",
            "time serie",
            "unmanned aerial vehicle",
            "video game",
        ],
    },
    {
        "id": 303577,
        "nombre": "Fernando Pescador Del Oso",
        "keywords": [
            "dsp",
            "hevc",
            "standard",
            "rvc-cal",
            "multicore",
            "video decoder",
            "encoding",
            "vvc",
            "versatile video coding",
            "decoder",
            "openmp",
            "fpga",
            "hardware architecture",
            "software optimization",
            "pipeline",
        ],
    },
    {
        "id": 307275,
        "nombre": "David Luengo García",
        "keywords": [
            "bayesian inference",
            "monte carlo method",
            "adaptive importance sampling",
            "population monte carlo",
            "electrocardiography",
            "multiple importance sampling",
            "adaptive mcmc",
            "compressive sensing",
            "ecg signal processing",
            "classifier chain",
            "multi-label classification",
            "sparse reconstruction",
            "delineation",
            "electrocardiogram (ecg)",
            "parallel chain",
        ],
    },
    {
        "id": 307625,
        "nombre": "Sara Lana Serrano",
        "keywords": [
            "information retrieval",
            "linguistic engineering",
            "image retrieval",
            "indexing",
            "text categorization",
            "domain-specific vocabulary",
            "thesauru",
            "relevance feedback",
            "topic expansion",
            "clef, 2008",
            "semantic expansion",
            "classification",
            "clef",
            "gazetteer",
            "geographic entity recognition",
            "geographical ir",
            "imageclef",
            "trie indexing",
            "action pattern",
            "axis",
            "context",
            "image annotation",
            "interface language",
            "irma code",
            "learning algorithm",
            "log file analysi",
            "logclef",
            "machine learning",
            "medical domain-specific vocabulary",
            "medical image",
            "native language",
            "nearest-neighbour",
            "ontology",
            "spatial retrieval",
            "the european library",
            "user language",
            "word sense disambiguation",
            "wordnet",
        ],
    },
    {
        "id": 308607,
        "nombre": "Carlos Badenes Olmedo",
        "keywords": [
            "knowledge graph",
            "document retrieval",
            "natural language processing",
            "public procurement",
            "linked data",
            "ontology",
            "antiretroviral drug",
            "hiv",
            "polypharmacy",
            "cord-1",
            "cord-19",
            "dynamic topic model",
            "topic interpretability",
            "topic labelling",
            "clustering",
            "document similarity",
            "hashing",
            "information search and retrieval",
            "open data",
        ],
    },
    {
        "id": 309146,
        "nombre": "Jose Ignacio Abalos Vazquez,",
        "keywords": [
            "architecture - spain",
            "architecture-spain",
            "barcelona ensanche",
            "community center",
            "cultural center",
            "social integration",
            "sustainability",
            "thermodynamic",
        ],
    },
    {
        "id": 313894,
        "nombre": "Mario Refoyo Lopez",
        "keywords": [],
    },
]

# Guardamos los PDIs en Qdrant
for pdi in pdis:
    index_pdi(client, my_collection, pdi)

In [ ]:
# Comprobamos que se han guardado correctamente y vemos los embeddings generados
print("PDIs indexados:")
records, next_page_offset = client.scroll(
    collection_name=my_collection,
    scroll_filter=None,  # Add filters here if needed
    limit=10,  # Number of records to fetch
    with_payload=True,  # Set to False to exclude metadata
    with_vectors=True,  # Set to True to include your embeddings
)

for record in records:
    print(
        f"Point ID: {record.id} - Name: {record.payload['nombre']} - Embeddings: {record.vector['keywords_arquitectura'].indices}"
    )

PDIs indexados:
Point ID: 261274 - Name: David Camacho Fernandez - Embeddings: [43067052, 74256320, 204076397, 277850205, 287978520, 524852419, 639288724, 705002106, 727709881, 818409348, 989116115, 1035425894, 1042626614, 1069987610, 1202643662, 1228389567, 1289061206, 1298296063, 1318831999, 1380609464, 1517563190, 1566116842, 1603994372, 1608777121, 1644170059, 1707350732, 1784532271, 1943344364, 2058513491, 2069342412]
Point ID: 303577 - Name: Fernando Pescador Del Oso - Embeddings: [198514941, 236605023, 639288724, 685567477, 705002106, 794129062, 964658953, 1065868547, 1090888193, 1150895764, 1416086225, 1427162981, 1438597426, 1464642605, 1471117264, 1530882065, 1696538840, 1853176582, 1864008405]
Point ID: 307275 - Name: David Luengo García - Embeddings: [41171975, 160759034, 252981213, 303380881, 310830179, 328952663, 440359455, 463787364, 540174517, 675534778, 710973585, 778874514, 780280564, 930174355, 989509788, 1059529610, 1152360124, 1335889639, 1418282702, 1431075457, 14

In [37]:
# Generadas por el LLM a partir de título y resumen de la idea
nuevas_keywords_a_buscar = [
    "Automated Question Generation",
    "Educational Question Generation",
    "AI-Powered Assessment Tools",
    "Intelligent Tutoring Systems Components",
    "Formative Assessment Automation",
    "Teaching Evaluation Technology",
    "Higher Education e-Assessment",
    "Artificial Intelligence in Education (AIEd)",
    "Machine Learning Applications in Education",
    "Adaptive Question Generation using LLMs",
    "Automatic Multiple Choice and Open Ended Question Generation",
    "Bloom's Taxonomy Integration in QG (Question Generation)",
    "Semantic Chunking for Question Generation from Text",
    "Control of Cognitive Difficulty Level in Automated Questions",
    "Web Service Deployment for Educational LLMs",
    "Clarity Evaluation of Automatically Generated Questions",
    "Pedagogical Alignment Assessment (LLMs)",
    "QuerIA System (Research Specific)",
    "Question Generation Prompt Engineering with Bloom's Taxonomy",
    "Large Language Models in Teacher Education",
    "Development and Deployment",
]


def normalize_keywords(keywords):
    return set(k.strip().lower() for k in keywords)


# Normalizamos las keywords
nuevas_keywords = normalize_keywords(nuevas_keywords_a_buscar)

# Vectorizamos y buscamos los PDIs más similares a las nuevas keywords
best_pdis = search_similar_pdis(client, my_collection, nuevas_keywords)
best_pdis

[ScoredPoint(id=307625, version=25, score=0.06921433, payload={'nombre': 'Sara Lana Serrano'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=308607, version=26, score=0.03610408, payload={'nombre': 'Carlos Badenes Olmedo'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=307275, version=24, score=0.02544977, payload={'nombre': 'David Luengo García'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=261274, version=22, score=0.019465718, payload={'nombre': 'David Camacho Fernandez'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=309146, version=27, score=0.019259155, payload={'nombre': 'Jose Ignacio Abalos Vazquez,'}, vector=None, shard_key=None, order_value=None)]

In [40]:
[pdi.score for pdi in best_pdis]

[0.06921433, 0.03610408, 0.02544977, 0.019465718, 0.019259155]

In [ ]:
collection_name = "mi_primera_coleccion"
lista_de_keywords = ["tecnologia", "moda", "hogar", "deportes"]


# Vector de prueba (ej: generado a partir de una consulta del usuario)
vector_consulta = [0.2, 0.7, 0.6, 0.2]

# Búsqueda simple por similitud
resultados = client.query_points(
    collection_name=collection_name,
    query=vector_consulta,
    limit=2,  # Traer los 2 resultados más cercanos
).points

print("--- Resultados de búsqueda simple ---")
for res in resultados:
    print(f"ID: {res.id}, Score: {res.score}, Metadata: {res.payload}")


# Búsqueda Avanzada aplicando un Filtro (solo productos de 'tecnologia')
resultados_filtrados = client.query_points(
    collection_name=collection_name,
    query=vector_consulta,
    query_filter=Filter(
        must=[FieldCondition(key="categoria", match=MatchValue(value="tecnologia"))]
    ),
    limit=2,
).points

print("\n--- Resultados aplicando filtro ('tecnologia') ---")
for res in resultados_filtrados:
    print(f"ID: {res.id}, Score: {res.score}, Metadata: {res.payload}")